<a href="https://colab.research.google.com/github/gauravraidata/LLL_Bots/blob/main/RAG_BOT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip uninstall -y langchain
!pip install langchain
!pip install langchain-huggingface
!pip install langchain-community
!pip install langchain-core
!pip install chromadb
!pip install sentence_transformers
!pip install huggingface_hub


Found existing installation: langchain 1.3.18
Uninstalling langchain-1.3.18:
  Successfully uninstalled langchain-1.3.18
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 73.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52

In [5]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.prompts import PromptTemplate
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.llms import HuggingFaceHub # New import for HuggingFaceHub
# PDF LOader
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
# semantic search
from langchain_classic.chains import RetrievalQA

/tmp/ipykernel_2256/3699400298.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings


In [6]:
# Load HuggingFace API key securely from Colab Secrets
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

In [7]:
#Step wise RAG pipeline development
import os
os.environ["HUGGINGFACEHUB_API_TOKEN"] = HF_TOKEN # Corrected environment variable name

LLM = ChatHuggingFace(llm=HuggingFaceEndpoint(repo_id="meta-llama/Llama-3.1-8B-Instruct", task="conversational")) # Specified task for conversational model

In [8]:
result = LLM.invoke("What is the capital of India?")
result

AIMessage(content='The capital of India is New Delhi.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 17, 'total_tokens': 26}, 'model_name': 'meta-llama/Llama-3.1-8B-Instruct', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a09edc-e293-71e0-8995-810dc88899df-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 17, 'output_tokens': 9, 'total_tokens': 26})

In [9]:
!pip install torch transformers==5.2.0 "qwen-vl-utils[decord]==0.0.14" \
  "sentence-transformers>=5.7.0" "accelerate>=1.1.0"


In [1]:
from langchain_huggingface import HuggingFaceEmbeddings

#need embedding model now
Emb = HuggingFaceEmbeddings(
            model_name="tencent/WeMM-Embedding-9B",
            model_kwargs={'device': 'cpu', 'trust_remote_code': True},
            encode_kwargs={'normalize_embeddings': True}
        )

model.safetensors: reconstructing file:   0%|          |  0.00B / 18.8GB            

model.safetensors: downloading bytes:           |  0.00B            

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/760 [00:00<?, ?it/s]

processor_config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

...

sentence_transformers.jinja:   0%|          | 0.00/1.45k [00:00<?, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


tokenizer_config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 20.0MB            

tokenizer.json: downloading bytes:           |  0.00B            

In [2]:
!pip install -U pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 14.1 MB/s eta 0:00:00


In [10]:
Doc = PyPDFLoader(
    "/content/Gaurav_CV.pdf"
).load()

# Create chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(Doc)

print(f"Number of pages: {len(Doc)}")
print(f"Number of chunks: {len(chunks)}")

# Inspect the first chunk
print(chunks[0].page_content)
print(chunks[0].metadata)

Number of pages: 2
Number of chunks: 7
GAURAV RAI
Senior Data Analyst — Data Scientist
Mumbai, India — 8080125694 — gauravraidata@gmail.com
linkedin.com/in/gauravraidata — kaggle.com/gauravrai2000 — github.com/gauravraidata
Professional Summary
Data professional with 4+ years of experience in BI reporting, predictive modeling, and machine
learning, currently pursuing an M.Tech in Artificial Intelligence at IIT Jodhpur. Skilled in SQL,
Python, and Power BI, with hands-on experience building ML/DL models, automating dashboards,
and integrating multi-platform data (GCP, Azure, SharePoint) to drive business decisions for en-
terprise clients including Walmart Inc. Recognized top performer with a track record of exceeding
SLA and ticket-resolution targets. Passionate about applying explainable AI and analytics to solve
real-world business problems.
Technical Skills
•SQL:SQL Server, BigQuery, MySQL, T-SQL, P-SQL, Stored Procedures, Window Functions,
SSMS
{'producer': 'pdfTeX-1.40.27', 'creat

In [11]:
#lets perform enbedding in chroma db

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=Emb,
    persist_directory="db"
)

In [12]:

retriver = vector_store.as_retriever(search_kwargs={"k": 1}, type = 'similarity')

In [13]:
retriver.invoke("what is gaurav's education?")

[Document(metadata={'source': '/content/Gaurav_CV.pdf', 'total_pages': 2, 'author': '', 'moddate': '2026-08-02T06:48:48+00:00', 'page': 0, 'creationdate': '2026-08-02T06:48:48+00:00', 'page_label': '1', 'subject': '', 'creator': 'LaTeX with hyperref', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'title': '', 'keywords': '', 'trapped': '/False', 'producer': 'pdfTeX-1.40.27'}, page_content='GAURAV RAI\nSenior Data Analyst — Data Scientist\nMumbai, India — 8080125694 — gauravraidata@gmail.com\nlinkedin.com/in/gauravraidata — kaggle.com/gauravrai2000 — github.com/gauravraidata\nProfessional Summary\nData professional with 4+ years of experience in BI reporting, predictive modeling, and machine\nlearning, currently pursuing an M.Tech in Artificial Intelligence at IIT Jodhpur. Skilled in SQL,\nPython, and Power BI, with hands-on experience building ML/DL models, automating dashboards,\nand integrating multi-platform data (GCP, A

In [16]:
#augmentation
prompt = PromptTemplate(
    template = """you are a helpful assistance for HR, answer the questions only from the provided context.
    if context is insufficient, just return answer not available

    {context}
    Question : {question}
    """,
    input_variables = ["context", "question"]
)

In [17]:
question = "what is gaurav's education?"
R_DOC = retriver.invoke(question)

In [25]:
#joining all the context from page content
def format_docs(R_DOC):
  context_text = "\n\n".join(doc.page_content for doc in R_DOC)
  return context_text

In [26]:
context_text = format_docs(R_DOC)

In [22]:
final_prompt = prompt.format(context = context_text, question = question)
final_prompt

"you are a helpful assistance for HR, answer the questions only from the provided context. \n    if context is insufficient, just return answer not available\n\n    GAURAV RAI\nSenior Data Analyst — Data Scientist\nMumbai, India — 8080125694 — gauravraidata@gmail.com\nlinkedin.com/in/gauravraidata — kaggle.com/gauravrai2000 — github.com/gauravraidata\nProfessional Summary\nData professional with 4+ years of experience in BI reporting, predictive modeling, and machine\nlearning, currently pursuing an M.Tech in Artificial Intelligence at IIT Jodhpur. Skilled in SQL,\nPython, and Power BI, with hands-on experience building ML/DL models, automating dashboards,\nand integrating multi-platform data (GCP, Azure, SharePoint) to drive business decisions for en-\nterprise clients including Walmart Inc. Recognized top performer with a track record of exceeding\nSLA and ticket-resolution targets. Passionate about applying explainable AI and analytics to solve\nreal-world business problems.\nTechni

In [24]:
#generating
answer = LLM.invoke(final_prompt)
answer.content

"Master's degree in Artificial Intelligence (M.Tech) from IIT Jodhpur"

In [27]:
#building the chain

from langchain_core.runnables import RunnablePassthrough, RunnableLambda, RunnableParallel


In [28]:
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()

In [29]:
parallel_chain = RunnableParallel(
    {
        "context": retriver | RunnableLambda(format_docs),
        "question": RunnablePassthrough()
    }
)

In [30]:
main_chain = parallel_chain | prompt | LLM | parser

In [31]:
main_chain.invoke("summarize the CV")

'Here is a summary of the CV:\n\n**Data Analyst & Project Manager with Machine Design Experience**\n\n* Proven experience in data analysis, reporting, and project management\n* Skilled in integrating datasets from various sources (Excel, Google Sheets, SharePoint, Azure) into Power BI\n* Strong problem-solving skills, with ability to resolve data source issues and automate reporting processes\n* Collaborative mindset, with experience working with international teams and clients\n* Project management experience, including developing project plans and timelines for clients\n* Background in machine design and analytical approaches, with ability to optimize performance and manufacturing efficiency.'